In [ ]:
# --- cell 00 ---
import pandas as pd
import ipaddress
from ipaddress import ip_address
from ipwhois import IPWhois
from ipwhois.exceptions import IPDefinedError, HTTPLookupError
import numpy as np
import os
import json
from functools import reduce
import html
import re
import socket
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display

# plotly 6 FigureWidget: after a redraw the front end can report a trace delta with
# no uid, and plotly's handler dies with KeyError: 'uid'. Fill the uid in by trace
# position, then run plotly's own handler. The wrapper goes inside plotly's registered
# handler (its .func) -- replacing the handler itself would unregister it, silently
# dropping every front-end change instead of fixing it. Guarded so re-running this cell
# doesn't double-wrap.

from plotly.basewidget import BaseFigureWidget
if not getattr(BaseFigureWidget, "_uid_patch", False):
    _traceDeltasHandler = BaseFigureWidget.__dict__["_handler_js2py_traceDeltas"]
    _orig_trace_deltas  = _traceDeltasHandler.func
    def _trace_deltas_with_uid(self, change):
        for i, d in enumerate((change["new"] or {}).get("trace_deltas") or []):
            if "uid" not in d and i < len(self.data):
                d["uid"] = self.data[i].uid
        return _orig_trace_deltas(self, change)
    _traceDeltasHandler.func = _trace_deltas_with_uid
    BaseFigureWidget._uid_patch = True

In [ ]:
# --- cell 01 ---
import glob, os
for f in sorted(glob.glob("*.parquet")):
    print(f"{os.path.getsize(f) / 1e6:8.1f} MB  {f}")


In [ ]:
# --- cell 02 ---
parquet_file = "vflowsAgg-637602092272-october-2026.parquet"
vflowsInput = pd.read_parquet(parquet_file)   # the pre-processor's vflowsAgg sessions
vflowsInput.shape

In [ ]:
# --- cell 03 ---
# Settings, plus the per-IP `public` table the Cymru cells below enrich. It is the
# pre-processor's own per-IP table, loaded from the <sessions file>-ips.parquet its cell 26
# writes next to the sessions file -- not rebuilt here from sessions. Columns:
#   ip, ip_int    the public IPv4 address, as text and as an integer
#   label         its tag-org (cloud service / known_ips.csv name, <NA> if none)
#   tag-cloud     the reference list it matched (aws, azure, ...), "known" when
#                 known_ips.csv names it, <NA> if neither
#   cloud         True when tag-cloud is a reference list (set, and not "known"); in_ref
#                 is the same flag under the name cells 24-26 use
#   has_tag_org   True when the IP arrived with a tag-org (a cloud list or known_ips.csv
#                 already names it); cell 15 leaves their label alone. Set here, before
#                 cell 15 fills label in from Cymru
#   skip_cymru    True when tag-cloud has any value (a reference list or "known") --
#                 cell 14 doesn't ask Cymru about it
#   known_asn, known_cc   asn / country code from known_ips.csv; cell 14 uses them in
#                 place of Cymru's
#   ref_prefix, aws_region, known, known_label   as the pre-processor set them
# Every public IP in the sessions must be in the IP file, or the two files don't belong
# together and the cell stops.
# Ingress / egress is not handled here: cells 19 / 32 use the log's own flow-direction
# field when the logs carry it (v5) and skip direction when they don't (v2/v3). It is
# never derived -- which end is public isn't a reliable guess.

CIDR_DIR             = "cidrs"
CYMRU_CACHE          = "cymru_cache.csv"
ALLOW_EXTERNAL_CYMRU = False   # True = query whois.cymru.com for uncached IPs

_need = ["srcaddr", "dstaddr", "src-int", "dst-int", "tag-src-public", "tag-dst-public"]
if _miss := [c for c in _need if c not in vflowsInput.columns]:
    raise KeyError(f"vflowsInput is missing {_miss} -- load a vflowsAgg parquet written by the current "
                   "vflows-pre-processor (re-run it through cell 26)")

IPS_FILE = os.path.splitext(parquet_file)[0] + "-ips.parquet"
if not os.path.exists(IPS_FILE):
    raise FileNotFoundError(f"{IPS_FILE} not found -- re-run vflows-pre-processor through cell 26, "
                            "which writes it next to the sessions file")
public = pd.read_parquet(IPS_FILE)
if _miss := [c for c in ("ip", "ip_int", "tag-org", "tag-cloud", "known", "known_asn", "known_cc")
             if c not in public.columns]:
    raise KeyError(f"{IPS_FILE} is missing {_miss} -- re-run vflows-pre-processor through cell 26")

# the two files must come from the same pre-processor run
_sessInts = np.union1d(vflowsInput.loc[vflowsInput["tag-src-public"].fillna(False).astype(bool), "src-int"].to_numpy("uint64"),
                       vflowsInput.loc[vflowsInput["tag-dst-public"].fillna(False).astype(bool), "dst-int"].to_numpy("uint64"))
_notInFile = np.setdiff1d(_sessInts, public["ip_int"].to_numpy("uint64"))
if len(_notInFile):
    raise ValueError(f"{len(_notInFile):,} public IPs in {parquet_file} aren't in {IPS_FILE} -- the files "
                     "are from different runs; re-run vflows-pre-processor through cell 26")

public = (public.rename(columns={"tag-org": "label"})
          .astype({"ip": str, "ip_int": "uint64", "label": object, "tag-cloud": object})
          .sort_values("ip_int", kind="stable")
          .reset_index(drop=True))
public["tag-cloud"] = public["tag-cloud"].astype("string").astype(object)   # parquet's None -> <NA>
public["cloud"]       = public["tag-cloud"].notna() & public["tag-cloud"].ne("known")
public["in_ref"]      = public["cloud"]
public["has_tag_org"] = public["label"].notna()
public["skip_cymru"]  = public["tag-cloud"].notna()

print(f"{len(public):,} public IPs from {IPS_FILE}, {int(public['has_tag_org'].sum()):,} with a tag-org "
      f"({int(public['cloud'].sum()):,} on a cloud list)")

In [ ]:
# --- cell 04 ---
# Bulk IP -> ASN / announced BGP prefix via Team Cymru, with a local cache.
# One TCP connection for the whole *missing* list instead of a per-IP RDAP call.
# Cached IPs are reused; only uncached ones go out. IPs with skip_cymru (cell 06a: any
# tag-cloud, i.e. a reference list or "known") aren't looked up -- the list or
# known_ips.csv already says who they are. known_ips.csv IPs take their asn / cc from the
# file (step 4); an entry that leaves them blank gets none.
# No internet / blocked egress -> warn, keep going with whatever is already cached;
# uncached IPs stay unresolved and are retried on a later run. A corrupt cache file
# is ignored and rebuilt.

_cy_cols = ["asn", "bgp_prefix", "cc", "registry", "alloc", "as_name"]
_CY_EMPTY = pd.DataFrame(columns=["ip", *_cy_cols])

def _clean_cymru_text(value, max_len=300):
    """Keep third-party text inert before caching, displaying, or exporting."""
    if pd.isna(value):
        return value
    value = re.sub(r"[\x00-\x1f\x7f-\x9f]", "", str(value))[:max_len].strip()
    return "'" + value if value.startswith(("=", "+", "-", "@")) else value

def _sanitize_cymru_frame(df):
    df = df.copy()
    for col in ["ip", *_cy_cols]:
        if col in df.columns:
            df[col] = df[col].map(_clean_cymru_text)
    return df

def cymru_bulk(ip_list, timeout=15):
    """Query Team Cymru. Returns a DataFrame on success, or None if the call failed."""
    ips = list(dict.fromkeys(map(str, ip_list)))
    if not ips:
        return _CY_EMPTY.copy()
    payload = "begin\nverbose\n" + "\n".join(ips) + "\nend\n"
    try:
        with socket.create_connection(("whois.cymru.com", 43), timeout=timeout) as s:
            s.sendall(payload.encode())
            buf = b""
            while chunk := s.recv(65536):
                buf += chunk
    except OSError as e:
        print(f"!! Team Cymru lookup failed ({type(e).__name__}: {e}) -> no new enrichment")
        return None
    rows = []
    for ln in buf.decode(errors="replace").splitlines():
        if "|" not in ln or ln.startswith(("Bulk", "Warning", "Error")):
            continue
        parts = [p.strip() for p in ln.split("|")]
        if parts[0] == "AS":                               # header
            continue
        asn, ip, prefix, cc, registry, alloc, as_name = (parts + [""] * 7)[:7]
        rows.append({"ip": ip, "asn": asn, "bgp_prefix": prefix, "cc": cc,
                     "registry": registry, "alloc": alloc, "as_name": as_name})
    return pd.DataFrame(rows, columns=["ip", *_cy_cols])

# 1. load cache (keep_default_na=False so Cymru's literal "NA" survives round-trips)
try:
    cymru_cache = (pd.read_csv(CYMRU_CACHE, dtype=str, keep_default_na=False)
                   if os.path.exists(CYMRU_CACHE) else _CY_EMPTY.copy())
except (OSError, ValueError) as _e:
    print(f"!! could not read {CYMRU_CACHE} ({_e}) -> starting a fresh cache")
    cymru_cache = _CY_EMPTY.copy()
cymru_cache = _sanitize_cymru_frame(cymru_cache)

# 2. look up only the IPs not skipped (skip_cymru) and not already cached
want    = pd.Series(public.loc[~public["skip_cymru"], "ip"].astype(str).unique())
missing = want[~want.isin(cymru_cache["ip"])].tolist()
print(f"{len(want)} IPs needed ({int(public['skip_cymru'].sum())} with a tag-cloud skipped, "
      f"{int(public['tag-cloud'].eq('known').sum())} of them known)  |  "
      f"{len(cymru_cache)} cached  |  {len(missing)} uncached")
if missing and not ALLOW_EXTERNAL_CYMRU:
    print("  external Team Cymru lookups disabled; using cache only")

if missing and ALLOW_EXTERNAL_CYMRU:
    fresh = cymru_bulk(missing)
    if fresh is None:                       # network failed -> leave cache untouched
        print("  cache unchanged; uncached IPs will retry next run")
    else:
        stubs = pd.DataFrame({"ip": [m for m in missing if m not in set(fresh["ip"])]})
        fresh = _sanitize_cymru_frame(fresh)
        stubs = _sanitize_cymru_frame(stubs)
        cymru_cache = (pd.concat([cymru_cache, fresh, stubs], ignore_index=True)
                         .fillna("")
                         .drop_duplicates(subset="ip", keep="first"))
        try:
            cymru_cache.to_csv(CYMRU_CACHE, index=False)
        except OSError as _e:
            print(f"!! could not write {CYMRU_CACHE} ({_e})")
        print(f"  +{len(fresh)} resolved, +{len(stubs)} unresolved  ->  {len(cymru_cache)} rows")

# 3. attach to public. "" means "no data" in every column -> NaN. Cymru's literal "NA"
#    means "no data" only in asn / bgp_prefix / as_name; in cc it is Namibia, so it stays.
_CY_NA_COLS = ["asn", "bgp_prefix", "as_name"]
cymru = cymru_cache[cymru_cache["ip"].isin(want)].copy()
if len(cymru):
    cymru[_cy_cols]    = cymru[_cy_cols].replace("", np.nan)
    cymru[_CY_NA_COLS] = cymru[_CY_NA_COLS].replace("NA", np.nan)
public = public.drop(columns=[c for c in _cy_cols if c in public.columns])   # re-run safe
public = public.merge(cymru, on="ip", how="left")

# 4. known_ips.csv asn / cc win over Cymru's -- they're your own, more specific facts
for _c, _k in (("asn", "known_asn"), ("cc", "known_cc")):
    _m = public[_k].notna()
    public[_c] = public[_c].astype(object)
    public.loc[_m, _c] = public.loc[_m, _k].astype(str)

print(f"{cymru['bgp_prefix'].nunique()} distinct BGP prefixes, "
      f"{cymru['asn'].nunique()} distinct ASNs for {len(cymru)} IPs")


In [ ]:
# --- cell 05 ---
# Fill the gaps: anything the CIDR lookups didn't cover gets labelled from Cymru.
# IPs that arrived with a tag-org (has_tag_org, cell 06a) are skipped (no owner, label
# left as is) -- cell 14 didn't look them up, and a cloud list or known_ips.csv names them.
# Fallback chain:  AS owner name  ->  AS<number>  ->  unrouted/<registry>  ->  unknown
# Re-run safe: rows already at "-"/""/"unknown"/NaN are re-evaluated.

# Cymru as_name is "HANDLE - Org Name, CC". The Org field is usually a company name
# but for some ASNs (older CN/IN/TW allocations) it's a street address or a generic
# backbone phrase. When it looks like that, fall back to HANDLE, which never is.

_ADDR_RE = re.compile(
    r"(Street|St\.| Road|Rd\.|Avenue| Ave| Way,|Floor| Fl\.|Bldg|Building|Room| Rm\.|"
    r"District|Zhonglu| Lu,| Jie |No\.\s*\d|\bRace Course\b|Blvd|Backbone|Provincial|"
    r"Business Group|Internet Backbone|Net of CT)", re.I)

def short_owner(as_name):
    if not isinstance(as_name, str) or not as_name.strip():
        return None
    handle, sep, rest = as_name.partition(" - ")
    handle = handle.strip()
    if not sep:                                    # not "HANDLE - ORG" shaped
        return as_name.rsplit(",", 1)[0].strip() or None
    org = rest.rsplit(",", 1)[0].strip()           # drop the trailing ", CC"
    if not org or len(org) < 3 or _ADDR_RE.search(org):
        return handle or None
    return org

# optional data-driven overrides (cidrs/org_overrides.csv: from,to) -- no code edits needed
_ovf = os.path.join(CIDR_DIR, "org_overrides.csv")
_overrides = (dict(pd.read_csv(_ovf, dtype=str).dropna().itertuples(index=False, name=None))
              if os.path.exists(_ovf) else {})

public["owner"] = public["as_name"].where(~public["has_tag_org"]).map(short_owner).replace(_overrides)

# build the fallback, tier by tier
_fb = public["owner"].copy()
_m  = _fb.isna() & public["asn"].notna()
_fb[_m] = "AS" + public.loc[_m, "asn"].astype(str)                    # named-less but routed
_m  = _fb.isna() & public["registry"].notna()
_fb[_m] = "unrouted/" + public.loc[_m, "registry"].astype(str)        # no BGP origin at all

_needs_fill = ~public["has_tag_org"] & (public["label"].isna() | public["label"].isin(["-", "", "unknown"]))
public["label"] = np.where(_needs_fill, _fb.fillna("unknown"), public["label"])

print(public["label"].value_counts().head(20).to_string())
print(f"\nstill unknown: {(public['label'] == 'unknown').sum()}")

In [ ]:
# --- cell 06 ---
# Resolve the 2-letter cc to a country name via cidrs/country_codes.csv (no library).

_ccf = os.path.join(CIDR_DIR, "country_codes.csv")
try:
    _cc_names = dict(pd.read_csv(_ccf, dtype=str, keep_default_na=False).itertuples(index=False, name=None))
except (OSError, ValueError) as _e:
    print(f"!! {_ccf} not loaded ({type(_e).__name__}: {_e})")
    _cc_names = {}

public["country"] = public["cc"].map(_cc_names)

_have = int(public["cc"].notna().sum())
_got  = int(public["country"].notna().sum())
_unmapped = sorted(set(public.loc[public["cc"].notna() & public["country"].isna(), "cc"]))
print(f"{_got} / {_have} IPs got a country name"
      + (f"   unmapped cc: {_unmapped}" if _unmapped else ""))
public["country"].value_counts().head(25)

In [ ]:
# --- cell 07 ---
# The IPs that only got a fallback label -- 'unrouted/<rir>' (no BGP origin) or
# 'AS<n>' (routed but unnamed) -- with how they actually show up in the flow data
# (fallback_view, kept for drill-down, not displayed). Prints each fallback kind with
# what it means, followed by its label / country rows, IP counts and traffic.

pd.set_option("display.max_rows",    10)
pd.set_option("display.max_columns", None)
pd.set_option("display.width",       None)

if "owner" not in public.columns or "country" not in public.columns:
    raise RuntimeError("public has no Cymru labels / countries yet -- run cells 14, 15 and 16 "
                       "after cell 06a (re-running 06a resets public)")

_fb_ips = set(public.loc[public["label"].str.match(r"^(unrouted/|AS\d)", na=False), "ip"])

_hit = vflowsInput[vflowsInput["srcaddr"].isin(_fb_ips) | vflowsInput["dstaddr"].isin(_fb_ips)].copy()
_hit["peer"] = _hit["srcaddr"].where(_hit["srcaddr"].isin(_fb_ips), _hit["dstaddr"])
_stats = _hit.groupby("peer").agg(
    flows=("bytes", "count"),
    bytes=("bytes", "sum"),
    pkts=("packets", "sum"),
    actions=("action", lambda s: "/".join(sorted(s.dropna().unique()))),
)

fallback_view = (
    public.loc[public["ip"].isin(_fb_ips),
               ["ip", "label", "asn", "bgp_prefix", "cc", "country", "registry", "alloc"]]
    .merge(_stats, left_on="ip", right_index=True, how="left")
    .sort_values(["label", "bytes"], ascending=[True, False])
)
_pairs = (fallback_view.assign(country=fallback_view["country"].fillna("(no country)"))
          .groupby(["label", "country"])
          .agg(ips=("ip", "size"), flows=("flows", "sum"), bytes=("bytes", "sum"),
               actions=("actions", lambda s: "/".join(sorted({a for v in s.dropna() for a in v.split("/")}))))
          .assign(mb=lambda d: (d["bytes"] / 1e6).round(2))
          .drop(columns="bytes")[["ips", "flows", "mb", "actions"]]
          .reset_index())
_isUnrouted = _pairs["label"].str.startswith("unrouted/")

print(f"{len(fallback_view)} public IPs got a fallback label -- Team Cymru had no owner name for them.")
print("Each row: label, country, how many of those IPs, and their sessions / MB / actions in the flows.")
for _sub, _head in (
        (_pairs[_isUnrouted],
         ["unrouted/<registry> -- the address is registered (AFRINIC, APNIC, ARIN, LACNIC or RIPE) but no",
          "network announces a route to it. Replies to it can't be delivered, so traffic from it is",
          "usually spoofed, scanning, or leftover address space."]),
        (_pairs[~_isUnrouted],
         ["AS<number> -- the address is routed, but its network has no registered name, so the label",
          "is just the AS number."])):
    print()
    print(f"{int(_sub['ips'].sum())} IPs: {_head[0]}")
    for _ln in _head[1:]:
        print(_ln)
    if len(_sub):
        for _ln in _sub.to_string(index=False).splitlines():
            print("    " + _ln)
    else:
        print("    (none)")

In [ ]:
# --- cell 08 ---
# Attach the per-IP enrichment to EVERY flow (ACCEPT and REJECT), both ends.
# `public` is the per-IP dimension table (Cymru fields + AWS-aware label); join it
# twice -- once on srcaddr, once on dstaddr -- as src_* / dst_* columns.
# Private addresses aren't in `public`, so their src_/dst_ fields stay NaN.

_enrich = ["label", "asn", "as_name", "cc", "country", "registry", "bgp_prefix", "cloud"]
_dim    = public.set_index("ip")[_enrich]

_pref = [f"{side}_{c}" for side in ("src", "dst") for c in _enrich]
vflows_enr = vflowsInput.drop(columns=[c for c in _pref if c in vflowsInput.columns])   # re-run safe
vflows_enr = (
    vflows_enr
    .merge(_dim.add_prefix("src_"), left_on="srcaddr", right_index=True, how="left")
    .merge(_dim.add_prefix("dst_"), left_on="dstaddr", right_index=True, how="left")
)

# Coverage of the public IPs: where each one's label came from (first match wins), and
# how many carry Cymru data / a country.
_own = ~public["has_tag_org"]
_src = pd.Series(np.select(
    [public["cloud"],
     public["tag-cloud"].eq("known"),
     _own & public["owner"].notna(),
     _own & public["owner"].isna() & public["asn"].notna(),
     _own & public["owner"].isna() & public["asn"].isna() & public["registry"].notna()],
    ["cloud list (tag-cloud)", "known_ips.csv", "Cymru owner name", "Cymru AS number only",
     "Cymru unrouted/<rir>"],
    default="not labelled"), index=public.index)
_n = len(public)
_cov = (_src.value_counts()
        .reindex(["cloud list (tag-cloud)", "known_ips.csv", "Cymru owner name",
                  "Cymru AS number only", "Cymru unrouted/<rir>", "not labelled"], fill_value=0)
        .rename("ips").to_frame()
        .assign(pct=lambda d: (100 * d["ips"] / max(_n, 1)).round(1)))
_lab = _n - int(_cov.loc["not labelled", "ips"])
print(f"{_n:,} public IPs: {_lab:,} labelled ({100 * _lab / max(_n, 1):.1f}%), "
      f"{_n - _lab:,} not ({100 * (_n - _lab) / max(_n, 1):.1f}%)")
print(_cov.to_string())
_cy = public[["asn", "registry", "cc"]].notna().any(axis=1)
print(f"Cymru data: {int(_cy.sum()):,} ({100 * _cy.mean():.1f}%)   |   "
      f"country: {int(public['country'].notna().sum()):,} ({100 * public['country'].notna().mean():.1f}%)")

In [ ]:
# --- cell 09 ---
# Every flow has exactly one public end -> collapse the two sides into one "peer"
# (the external party), taking whichever side isn't NaN. Ingress / egress is the
# log's own `flow-direction` field, present only in v5 logs (skipped when absent).
# ACCEPT / REJECT stays in `action` -- the cells
# below filter on it (e.g. .query('action == "ACCEPT"')) rather than keeping a copy.

vflows_enr["peer_label"]   = vflows_enr["src_label"].combine_first(vflows_enr["dst_label"])
vflows_enr["peer_asn"]     = vflows_enr["src_asn"].combine_first(vflows_enr["dst_asn"])
vflows_enr["peer_cc"]      = vflows_enr["src_cc"].combine_first(vflows_enr["dst_cc"])
vflows_enr["peer_country"] = vflows_enr["src_country"].combine_first(vflows_enr["dst_country"])
vflows_enr["peer_cloud"]   = (vflows_enr["src_cloud"]
                              .combine_first(vflows_enr["dst_cloud"])
                              .eq(True))   # NaN (private side) -> not cloud
vflows_enr["peer_addr"]    = np.where(vflows_enr["src_label"].notna(),
                                      vflows_enr["srcaddr"], vflows_enr["dstaddr"])

print(vflows_enr["peer_label"].isna().sum(), "rows with no peer label (action=None records)")
print(f"{int(vflows_enr['action'].eq('ACCEPT').sum()):,} ACCEPT of {len(vflows_enr):,} flows"
      + ("   |   " + "  ".join(f"{k} {v:,}" for k, v in
                              vflows_enr["flow-direction"].value_counts(dropna=False).items())
         if "flow-direction" in vflows_enr.columns else ""))
vflows_enr.query('action == "ACCEPT"')["peer_label"].value_counts().head(15)

In [ ]:
# --- cell 10 ---
# Toggle action (ACCEPT/REJECT) x direction (ingress/egress) on a live map. Direction
# is the log's own flow-direction field; without it (v2/v3 logs) the direction boxes
# are greyed out and every direction is shown.
# The figure is a FigureWidget updated IN PLACE via batch_update() -- rendering it
# inside an Output widget and redrawing kills plotly's hover/zoom/pan. Text panes are
# HTML widgets whose .value is *assigned*, not Outputs printed into: plotly fires
# on_click more than once per click, and assignment can't accumulate. Filters are
# precomputed boolean columns. JupyterLab only: VS Code's plotly renderer has no geo.

_a3 = dict(pd.read_csv(os.path.join(CIDR_DIR, "country_alpha3.csv"),
                       dtype=str, keep_default_na=False)   # 'NA' is Namibia, not null
             [["cc", "alpha3"]].itertuples(index=False, name=None))
_cc_of      = {v: k for k, v in _a3.items()}      # alpha3 -> cc, for click handling
_NO_POLYGON = {"HKG", "MAC"}                      # in the data, no polygon in the geo layer
_CSS        = "font:13px system-ui,sans-serif;margin:2px 0 6px"
_TITLE_CSS  = "font:15px system-ui,sans-serif;margin:8px 0 2px"

def _clean_widget_text(value, max_len=300):
    if pd.isna(value):
        return ""
    return re.sub(r"[\x00-\x1f\x7f-\x9f]", "", str(value))[:max_len].strip()

def _html_text(value, max_len=300):
    return html.escape(_clean_widget_text(value, max_len=max_len), quote=True)

_noncloud = ~vflows_enr["peer_cloud"]
_HAS_DIR = "flow-direction" in vflows_enr.columns and bool(vflows_enr["flow-direction"].notna().any())
_noDir   = pd.Series(False, index=vflows_enr.index)                    # placeholder, never used to filter
_mask = {"ACCEPT":  vflows_enr["action"].eq("ACCEPT"),
         "REJECT":  vflows_enr["action"].eq("REJECT"),
         "ingress": vflows_enr["flow-direction"].eq("ingress") if _HAS_DIR else _noDir,
         "egress":  vflows_enr["flow-direction"].eq("egress")  if _HAS_DIR else _noDir}

_cb     = {k: widgets.Checkbox(value=True, description=k, indent=False,
                               disabled=(k in ("ingress", "egress") and not _HAS_DIR),
                               layout=widgets.Layout(width="105px")) for k in _mask}
_title  = widgets.HTML()                          # map title, kept out of the figure so plotly's
_note   = widgets.HTML()                          # toolbar (top right) can't cover it
_detail = widgets.HTML()
_state  = {"mask": None}

# Fixed colour range so a shade means the same MB across selections.
_figw = go.FigureWidget(go.Choropleth(
    locations=[], z=[], locationmode="ISO-3", colorscale="OrRd", zmin=-1, zmax=4,
    marker_line_color="#888", marker_line_width=0.4,
    colorbar=dict(title="MB", tickvals=[-1, 0, 1, 2, 3, 4],
                  ticktext=["0.1", "1", "10", "100", "1K", "10K"],
                  lenmode="pixels", len=240, thickness=14,       # fixed size: the map keeps its own
                  y=0.5, yanchor="middle"),                       # aspect, so a full-height bar overshoots
    hovertemplate="<b>%{customdata[0]}</b><br>%{customdata[1]:,.1f} MB"
                  "<br>%{customdata[2]:,} flows<extra></extra>"))
_figw.update_geos(showcountries=True, countrycolor="#888",
                  lataxis_range=[-58, 85])                        # no Antarctica: wider map, less empty space
_figw.update_layout(margin=dict(l=0, r=0, t=10, b=0), autosize=True, height=380,
                    modebar_remove=["select2d", "lasso2d"])       # map uses clicks, not box / lasso select


def _blank(msg):
    with _figw.batch_update():
        _figw.data[0].locations, _figw.data[0].z = [], []
    _title.value = f"<div style='{_TITLE_CSS}'>{_html_text(msg)}</div>"
    _note.value = _detail.value = ""


def _update(_=None):
    acts = [k for k in ("ACCEPT", "REJECT") if _cb[k].value]
    dirs = [k for k in ("ingress", "egress") if _cb[k].value] if _HAS_DIR else []
    _detail.value = ""
    if not acts or (_HAS_DIR and not dirs):
        _blank("pick at least one action" + (" and one direction" if _HAS_DIR else ""))
        return

    m = _noncloud & reduce(lambda a, b: a | b, (_mask[k] for k in acts))
    if _HAS_DIR:
        m &= reduce(lambda a, b: a | b, (_mask[k] for k in dirs))
    _state["mask"] = m
    agg = (vflows_enr[m].groupby(["peer_cc", "peer_country"], dropna=False)
           .agg(flows=("bytes", "count"), bytes=("bytes", "sum"))
           .reset_index()
           .assign(mb=lambda d: (d["bytes"] / 1e6).round(1)))
    sel = f"{' + '.join(acts)}  /  {' + '.join(dirs) if _HAS_DIR else 'all directions'}"
    if agg.empty:
        _blank(f"no non-cloud flows for {sel}")
        return

    g  = agg.assign(alpha3=lambda d: d["peer_cc"].map(_a3),
                    log_mb=lambda d: np.log10(d["mb"].clip(lower=0.1)))
    gm = g.dropna(subset=["alpha3"])
    with _figw.batch_update():
        t = _figw.data[0]
        t.locations  = gm["alpha3"].tolist()
        t.z          = gm["log_mb"].tolist()
        t.customdata = gm.assign(peer_country=lambda d: d["peer_country"].map(_html_text))[["peer_country", "mb", "flows"]].values.tolist()
    _title.value = (f"<div style='{_TITLE_CSS}'><b>Non-cloud peers</b> — {_html_text(sel)}"
                    f" &nbsp;({agg['flows'].sum():,} flows, {agg['bytes'].sum()/1e6:,.1f} MB)</div>")

    off = g[g["alpha3"].isin(_NO_POLYGON) | g["alpha3"].isna()]
    off = ", ".join(f"{'unresolved cc' if pd.isna(r.peer_country) else _html_text(r.peer_country)}"
                    f" {r.mb:,.1f} MB" for r in off.itertuples()) if len(off) else ""
    _note.value = (f"<div style='{_CSS}'>click a country for its peers"
                   + (f"<br><span style='color:#a33'>not on the map "
                      f"(no polygon / unresolved cc): {off}</span>" if off else "")
                   + "</div>")


def _fmt_size(b):
    for u in ("B", "KB", "MB", "GB"):
        if abs(b) < 1000 or u == "GB":
            return f"{b:,.0f} {u}" if u == "B" else f"{b:,.1f} {u}"
        b /= 1000


def _html_table(df, right=()):
    """df -> HTML table with spacing and alignment set on every cell, so it reads the same
    in any notebook front end (some drop table stylesheets). Columns in `right` align right."""
    def _cell(tag, col, val, extra=""):
        side = "right" if col in right else "left"
        return (f"<{tag} style='padding:3px 14px;text-align:{side};white-space:nowrap;{extra}'>"
                f"{_html_text(val)}</{tag}>")
    head = "".join(_cell("th", c, c, "border-bottom:1px solid #888;font-weight:600") for c in df.columns)
    rows = "".join(
        f"<tr style='background:{'rgba(128,128,128,.10)' if i % 2 else 'transparent'}'>"
        + "".join(_cell("td", c, v) for c, v in zip(df.columns, r)) + "</tr>"
        for i, r in enumerate(df.itertuples(index=False)))
    return (f"<table style='font:12px ui-monospace,monospace;border-collapse:collapse;margin-top:6px'>"
            f"<thead><tr>{head}</tr></thead><tbody>{rows}</tbody></table>")


def _on_click(trace, points, state):
    if not points.point_inds:
        return
    cc  = _cc_of.get(trace.locations[points.point_inds[0]])
    sub = vflows_enr[_state["mask"] & vflows_enr["peer_cc"].eq(cc)]
    if sub.empty:
        _detail.value = f"<div style='{_CSS}'>no flows for {_html_text(cc)} in this selection</div>"
        return
    top = (sub.groupby(["peer_label", "protocol", "dstport"])
              .agg(flows=("bytes", "count"), bytes=("bytes", "sum"))
              .sort_values("bytes", ascending=False)
              .head(25).reset_index())
    show = pd.DataFrame({"peer":     top["peer_label"],
                         "protocol": top["protocol"],
                         "dst port": top["dstport"],
                         "flows":    top["flows"].map("{:,}".format),
                         "data":     top["bytes"].map(_fmt_size)})
    _detail.value = (
        f"<div style='{_CSS}'><b>{_html_text(sub['peer_country'].iloc[0])}</b> — {len(sub):,} flows, "
        f"{_fmt_size(sub['bytes'].sum())}, {sub['peer_addr'].nunique():,} peer IPs"
        + _html_table(show, right=("protocol", "dst port", "flows", "data"))
        + "</div>")


_figw.data[0].on_click(_on_click)
for _c in _cb.values():
    _c.observe(_update, names="value")

_update()
display(widgets.HBox([widgets.HTML("<b>action:</b>"), _cb["ACCEPT"], _cb["REJECT"],
                      widgets.HTML("&nbsp;<b>direction:</b>" + ("" if _HAS_DIR else " (not in these logs)")),
                      _cb["ingress"], _cb["egress"]]),
        _title, _figw, _note, _detail)

In [ ]:
# --- cell 10a ---
# US non-cloud peers as a treemap: one box per peer_label, sized and shaded by bytes (same
# MB colour scale as the map). Only peers with no tag-cloud at all -- unlike cell 10, the
# known_ips.csv IPs (tag-cloud "known") are left out too. Its own action / direction
# toggles, working as in cell 10. Click a box for the ASNs behind that label. "US" is Cymru's cc -- where the prefix is
# registered, not where the host sits. Names here are _us_-prefixed: cell 10's callbacks
# look up _figw, _state, ... by name when they fire, so reusing those names would break
# the map. Uses cell 10's _mask, _HAS_DIR and HTML helpers -- run it first.

_us_pfx    = vflows_enr["src_bgp_prefix"].combine_first(vflows_enr["dst_bgp_prefix"])
_us_asname = vflows_enr["src_as_name"].combine_first(vflows_enr["dst_as_name"])
_us_tagged = public.loc[public["tag-cloud"].notna(), "ip"]
_us_base   = vflows_enr["peer_cc"].eq("US") & ~vflows_enr["peer_addr"].isin(_us_tagged)

_us_cb     = {k: widgets.Checkbox(value=True, description=k, indent=False,
                                  disabled=(k in ("ingress", "egress") and not _HAS_DIR),
                                  layout=widgets.Layout(width="105px")) for k in _mask}
_us_title  = widgets.HTML()
_us_note   = widgets.HTML()
_us_detail = widgets.HTML()
_us_state  = {"mask": None, "labels": []}   # labels: raw peer_label per box, for clicks
_US_MAX_ROWS = 50                            # detail table rows (ASN + CIDR pairs)

_us_figw = go.FigureWidget(go.Treemap(
    labels=[], parents=[], values=[],
    marker=dict(colors=[], colorscale="OrRd", cmin=-1, cmax=4, showscale=True,
                line=dict(color="#888", width=0.4),
                colorbar=dict(title="MB", tickvals=[-1, 0, 1, 2, 3, 4],
                              ticktext=["0.1", "1", "10", "100", "1K", "10K"],
                              lenmode="pixels", len=240, thickness=14)),
    texttemplate="<b>%{label}</b><br>%{customdata[0]}",
    hovertemplate="<b>%{label}</b><br>%{customdata[0]}<br>%{customdata[1]:,} flows"
                  "<br>%{customdata[2]:,} ASNs<extra></extra>",
    pathbar_visible=False))
_us_figw.update_layout(margin=dict(l=0, r=0, t=10, b=0), autosize=True, height=480)


def _us_blank(msg):
    with _us_figw.batch_update():
        t = _us_figw.data[0]
        t.labels, t.parents, t.values, t.marker.colors = [], [], [], []
    _us_state["labels"] = []
    _us_title.value = f"<div style='{_TITLE_CSS}'>{_html_text(msg)}</div>"
    _us_note.value = _us_detail.value = ""


def _us_update(_=None):
    acts = [k for k in ("ACCEPT", "REJECT") if _us_cb[k].value]
    dirs = [k for k in ("ingress", "egress") if _us_cb[k].value] if _HAS_DIR else []
    _us_detail.value = ""
    if not acts or (_HAS_DIR and not dirs):
        _us_blank("pick at least one action" + (" and one direction" if _HAS_DIR else ""))
        return

    m = _us_base & reduce(lambda a, b: a | b, (_mask[k] for k in acts))
    if _HAS_DIR:
        m &= reduce(lambda a, b: a | b, (_mask[k] for k in dirs))
    _us_state["mask"] = m
    agg = (vflows_enr[m].groupby("peer_label")
           .agg(flows=("bytes", "count"), bytes=("bytes", "sum"), asns=("peer_asn", "nunique"))
           .sort_values("bytes", ascending=False)
           .reset_index())
    sel = f"{' + '.join(acts)}  /  {' + '.join(dirs) if _HAS_DIR else 'all directions'}"
    if agg.empty:
        _us_blank(f"no US non-cloud flows for {sel}")
        return

    with _us_figw.batch_update():
        t = _us_figw.data[0]
        t.labels        = agg["peer_label"].map(_html_text).tolist()
        t.parents       = [""] * len(agg)
        t.values        = agg["bytes"].tolist()
        t.marker.colors = np.log10((agg["bytes"] / 1e6).clip(lower=0.1)).tolist()
        t.customdata    = list(zip(agg["bytes"].map(_fmt_size), agg["flows"], agg["asns"]))
    _us_state["labels"] = agg["peer_label"].tolist()
    _us_title.value = (f"<div style='{_TITLE_CSS}'><b>US non-cloud peers</b> — {_html_text(sel)}"
                       f" &nbsp;({agg['flows'].sum():,} flows, {_fmt_size(agg['bytes'].sum())},"
                       f" {len(agg):,} peer labels)</div>")
    _us_note.value = f"<div style='{_CSS}'>click a box for its ASNs</div>"


def _us_on_click(trace, points, state):
    # the boxes share an implicit root (parent "") that has no point index -- a click on it
    # (gaps, edges) arrives as None, so only a real box index is used
    i = points.point_inds[0] if points.point_inds else None
    if not isinstance(i, int) or not 0 <= i < len(_us_state["labels"]):
        return
    lab = _us_state["labels"][i]
    sub = vflows_enr[_us_state["mask"] & vflows_enr["peer_label"].eq(lab)]
    if sub.empty:
        _us_detail.value = f"<div style='{_CSS}'>no flows for {_html_text(lab)} in this selection</div>"
        return
    # one row per ASN + CIDR; ASNs ordered by their total bytes, CIDRs by bytes within each
    g = (sub.assign(ASN=sub["peer_asn"].map(lambda a: "" if pd.isna(a) else f"AS{int(float(a))}"),
                    as_name=_us_asname.fillna(""), bgp_prefix=_us_pfx.fillna(""))
            .groupby(["ASN", "as_name", "bgp_prefix"])
            .agg(ips=("peer_addr", "nunique"), flows=("bytes", "count"), bytes=("bytes", "sum"))
            .reset_index())
    g["asn_bytes"] = g.groupby(["ASN", "as_name"])["bytes"].transform("sum")
    g = g.sort_values(["asn_bytes", "ASN", "bytes"], ascending=[False, True, False]).reset_index(drop=True)
    n_asn, n_pfx = g[["ASN", "as_name"]].drop_duplicates().shape[0], len(g)
    g = g.head(_US_MAX_ROWS)
    first = ~g.duplicated(["ASN", "as_name"])               # ASN / name only on a group's first row
    show = pd.DataFrame({"ASN":      g["ASN"].where(first, ""),
                         "AS name":  g["as_name"].where(first, ""),
                         "CIDR":     g["bgp_prefix"],
                         "peer IPs": g["ips"].map("{:,}".format),
                         "flows":    g["flows"].map("{:,}".format),
                         "data":     g["bytes"].map(_fmt_size)})
    more = (f"<div style='color:#a33;margin-top:4px'>first {_US_MAX_ROWS} of {n_pfx:,} CIDRs shown</div>"
            if n_pfx > _US_MAX_ROWS else "")
    _us_detail.value = (
        f"<div style='{_CSS}'><b>{_html_text(lab)}</b> — {len(sub):,} flows, "
        f"{_fmt_size(sub['bytes'].sum())}, {sub['peer_addr'].nunique():,} peer IPs, "
        f"{n_asn:,} ASN{'s' if n_asn != 1 else ''}, {n_pfx:,} CIDR{'s' if n_pfx != 1 else ''}"
        + _html_table(show, right=("peer IPs", "flows", "data"))
        + more + "</div>")


_us_figw.data[0].on_click(_us_on_click)
for _c in _us_cb.values():
    _c.observe(_us_update, names="value")

_us_update()
display(widgets.HBox([widgets.HTML("<b>action:</b>"), _us_cb["ACCEPT"], _us_cb["REJECT"],
                      widgets.HTML("&nbsp;<b>direction:</b>" + ("" if _HAS_DIR else " (not in these logs)")),
                      _us_cb["ingress"], _us_cb["egress"]]),
        _us_title, _us_figw, _us_note, _us_detail)

In [ ]:
# --- cell 11 ---
# Save the enriched sessions -- vflows_enr: vflowsInput plus the label / Cymru / peer
# columns from cells 18-19 -- to vflows-<account-id>-<year>.parquet in OUT_DIR.
# Account and year come from the data, not the clock: several accounts -> ids joined
# with "+"; sessions spanning several years (start_time / end_time) -> "<first>-<last>".
# Re-running overwrites the file.

OUT_DIR = "."

if "vflows_enr" not in globals():
    raise NameError("vflows_enr not defined -- run cells 18 and 19 first")
if _miss := [c for c in ("account-id", "start_time", "end_time") if c not in vflows_enr.columns]:
    raise KeyError(f"vflows_enr is missing {_miss} -- can't name the file")

_accts = sorted(vflows_enr["account-id"].dropna().astype(str).str.strip().unique())
_accts = [a for a in _accts if a and a != "-"]          # NODATA / SKIPDATA rows can carry "-"
if not _accts:
    raise ValueError("no account-id values in vflows_enr -- can't name the file")
_years = pd.concat([vflows_enr["start_time"], vflows_enr["end_time"]]).dropna().dt.year
if _years.empty:
    raise ValueError("no start_time / end_time values in vflows_enr -- can't name the file")
_y0, _y1 = int(_years.min()), int(_years.max())

OUT_ENR_PATH = os.path.join(OUT_DIR, f"vflows-enriched-{'+'.join(_accts)}-{_y0 if _y0 == _y1 else f'{_y0}-{_y1}'}.parquet")
vflows_enr.to_parquet(OUT_ENR_PATH, index=False)
print(f"wrote {len(vflows_enr):,} sessions x {vflows_enr.shape[1]} columns -> {OUT_ENR_PATH}  "
      f"({os.path.getsize(OUT_ENR_PATH) / 1e6:,.1f} MB)")